In [1]:
import matplotlib
import platform
print ("Operating system: ", platform.system())
if "Linux" in platform.system():
    %matplotlib tk
else:
    %matplotlib qt

    
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

#
import matplotlib.pyplot as plt
%autosave 180
%load_ext autoreload
%autoreload 2
import numpy as np

#
import scipy
import os
import time

#
import pickle 


#
from calibration.CalibrationTools import (CalibrationTools, get_binary_std_map, get_rois_stardist2d, get_img_std,
                                          save_calibration_data, save_ca_mask)

from drift.drift import (make_template, compute_drift_multi_frames, correct_drift, 
                         correct_drift_single_frame, template_generation, 
                         plot_mean_vs_template, make_motion_template_and_correct_data)

from utils.utils import smooth_ca_time_series, compute_dff0



Operating system:  Linux


Autosaving every 180 seconds


2023-03-04 10:15:43.850066: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-03-04 10:15:43.900375: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2023-03-04 10:15:43.902144: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /home/cat/.conda/envs/bmi/lib/python3.8/site-packages/cv2/../../lib64:
2023-03-04 10:15:43.902

In [8]:
#######################################################################
########### LOAD PRE-BMI DATA (e.g. 10-15mins recording) ##############
#######################################################################

#
#fname = r'F:\bmi\andres\20230115\M1\calibration\Image_001_001.raw'
fname = '/media/cat/4TB/donato/DON-010798/day0/calibration/Image_001_001.raw'

# 
bmi_c = CalibrationTools(fname)

#
bmi_c.smooth_ca_time_series = smooth_ca_time_series
bmi_c.compute_dff0 = compute_dff0

#
bmi_c.subsample = 10 # for std computation downsample to every N'th frame; the more frames the better the rois;
                  #   TODO: use correlation instead?! might be much faster; it is fast in other implemenations
    
print ("DONE...")

memmap :  (27000, 512, 512)
DONE...


In [9]:
##################################################################
############### MOTION CORRECTION STEP ###########################
##################################################################
# 
print ("Skipping template makgin step: ")
#bmi_c.template = np.mean(bmi_c.data[::100],axis=0)
bmi_c.template = np.mean(bmi_c.data[:1000],axis=0)
    
print ("DONE...")

Skipping template makgin step: 
DONE...


In [11]:
#################################################################
########### MAKE BINARY MAP FOR CELL REGISTRATOIN ###############
#################################################################

# convert float map to binary map using GUI
vmax = 1500
smin, smax = get_binary_std_map(bmi_c.template,vmax)
print ("...DONE...")


...DONE...


In [12]:
##################################################
############# MAKE IMAGE STD #####################
##################################################
bmi_c, img_std = get_img_std(smin, smax, bmi_c.template, bmi_c)
print ("DONE...")

max proj values (vmin, vmax):  651.590457256461 1083.9960238568588
DONE...


In [13]:
#########################################################
########### GENERATE CELL SEGMENTATION ##################
#########################################################
min_size_roi = 100  #<---- increase to exclude really small cells
max_size_roi = 800  #<----- decrease to exclude multi-cell artificats
bmi_c.roi_centres, bmi_c.footprints = get_rois_stardist2d(img_std,
                                               min_size_roi,
                                               max_size_roi,
                                               fname)



There are 4 registered models for 'StarDist2D':

Name                  Alias(es)
────                  ─────────
'2D_versatile_fluo'   'Versatile (fluorescent nuclei)'
'2D_versatile_he'     'Versatile (H&E nuclei)'
'2D_paper_dsb2018'    'DSB 2018 (from StarDist 2D paper)'
'2D_demo'             None
None
Found model '2D_versatile_fluo' for 'StarDist2D'.
Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.479071, nms_thresh=0.3.
1/1 [==============================] - 0s 103ms/step


100%|██████████████████████████████████████████████████████████████████████████████| 111/111 [00:00<00:00, 192.32it/s]

idx:  33
saving:  /media/cat/4TB/donato/DON-010798/day0/day0_ca_mask.npz
